<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-02/NB02_search_and_planning.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 02: Problem Solving by Search: Routes, Plans and Paths

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: September 2026.</sub>

</div>

## How to use this notebook

This notebook is a self-contained version of the week: The lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Each new piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt.

The animations of the lecture run on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-02/lecture.html), and the interactive lab with the self-assessment is available at <https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-02/lab.html>. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## Overview

Many engineering decisions are sequences: the route of a truck, the order of drilling holes, the moves of a crane or the path of a pipeline across a valley. Search turns such decisions into the exploration of a state space and finds a sequence of actions that reaches a goal at least cost [1, 2]. This week formulates problems as state spaces, compares uninformed search with heuristic search, proves why A* returns optimal paths when its heuristic never overestimates, and applies the ideas to path planning for vehicles on real terrain near Isparta [3, 4, 5].

**Estimated study time:** 8 to 10 hours.

## Learning outcomes

By the end of the week, students are expected to formulate an engineering task as a search problem with states, actions, transition model, goal test and path cost, to trace breadth-first search, uniform-cost search and A* by hand, to judge whether a heuristic is admissible and consistent, to explain the effect of a heuristic on the number of expanded nodes, and to implement grid path planning with terrain costs in Python.

## Python in this week

The lecture ends with Python step 2: Lists, tuples and loops. It covers lists, tuples, for and while loops, sets and a first queue, applied to routes and grid maps like those searched in this week [6]. The hands-on part of the notebook opens with the same step as runnable cells and short exercises with immediate feedback, and its later sections mark further constructs as Python moves.

## Problems as state spaces

A search problem has five parts [1]. The initial state describes where the agent starts. The actions available in a state say what the agent can do. The transition model gives the state that results from an action. The goal test decides whether a state solves the problem, and the path cost adds up the cost of the actions along a sequence. A solution is a sequence of actions from the initial state to a goal state, and an optimal solution has the lowest path cost among all solutions.

The formulation is an engineering decision in its own right. For a haul truck in an open-pit mine, a state can be a node of the road network, and the path cost can be travel time, fuel or tyre wear. For a mobile robot in a factory, the state can be a cell of an occupancy grid, which divides the floor into squares that are free or blocked [4]. For a crane that stacks containers, the state is the arrangement of all containers, and the number of states grows combinatorially. A good formulation keeps only the details that matter for the decision: A route planner for a truck does not need the colour of the truck, but it may need the slope of every road segment.


<details>
<summary><b>Check your understanding.</b> A planner chooses the order in which a laser cutter visits 60 contours on a steel sheet. What is a natural state in the formulation?</summary>

A. The colour of the steel sheet  
B. The set of contours already cut together with the current position of the cutting head  
C. The electrical power of the laser only  
D. The final cutting plan

**Answer: B.** The future cost depends on which contours remain and where the head is, so both belong in the state.

</details>

## Uninformed search: Breadth first, depth first and uniform cost

Uninformed strategies know nothing about the goal beyond the goal test. They differ only in the order in which they expand nodes of the search tree. Breadth-first search expands the shallowest node first by keeping the frontier in a first-in first-out queue. It finds the solution with the fewest actions, which is optimal when every action has the same cost. Depth-first search expands the deepest node first with a last-in first-out stack, needs little memory, but may wander along an infinite branch and does not guarantee the shortest solution. Iterative deepening repeats depth-limited search with growing limits and combines the memory use of depth-first search with the completeness of breadth-first search [7].

When actions have different costs, the fewest actions is not the cheapest path. Uniform-cost search expands the node with the lowest path cost so far, which is Dijkstra's algorithm on an explicit graph [8]. It is optimal for non-negative costs, because a node is expanded only when no cheaper route to it can exist.

The price of uninformed search is growth. With branching factor b and solution depth d, breadth-first search may generate on the order of b to the power d nodes. On a grid with four moves and a goal forty steps away, blind search explores almost every reachable cell within that radius. The number becomes unmanageable for combinatorial problems such as sequencing: Twenty jobs can be ordered in more than two quintillion ways.


![Growth of the number of nodes in a complete search tree with branching factor b. Uninformed search must be guided or limited when b or d is large [1].](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-02/figures/w02_fig1.png)

*Figure 2.1. Growth of the number of nodes in a complete search tree with branching factor b. Uninformed search must be guided or limited when b or d is large [1].*



<details>
<summary><b>Check your understanding.</b> Road segments of a network have different travel times. Which strategy returns the fastest route without extra knowledge about the goal?</summary>

A. Breadth-first search  
B. Depth-first search  
C. Uniform-cost search  
D. Random walk

**Answer: C.** Uniform-cost search, Dijkstra's algorithm, expands the node with the lowest accumulated cost, so it is optimal for non-negative costs.

</details>

## Informed search: Heuristics and A*

A heuristic function h(n) estimates the cost from node n to the nearest goal. Greedy best-first search expands the node with the smallest h and often reaches a goal quickly, but it can be misled into long detours. A* combines the cost already paid, g(n), with the estimate, and expands the node with the smallest f(n) = g(n) + h(n) [3].

Hart, Nilsson and Raphael showed that A* returns an optimal path if the heuristic is admissible, which means it never overestimates the true remaining cost [3]. The straight-line distance is admissible for road travel, because no road is shorter than a straight line. On a grid with four moves, the Manhattan distance, the sum of horizontal and vertical offsets, is admissible. A heuristic is consistent when its estimate never drops by more than the cost of one step. Consistency implies admissibility and guarantees that A* never needs to reopen a node [2].

Better heuristics save work. If one admissible heuristic is always at least as large as another, A* with the larger one expands no more nodes. The zero heuristic turns A* into uniform-cost search, and a perfect heuristic walks straight to the goal. Weighted A* multiplies the heuristic by a factor larger than one: It usually expands far fewer nodes and returns a path whose cost is at most that factor times the optimum, which is a common engineering compromise when planning must meet a deadline.


> **Animation: Breadth-first search and A* on the same map.** Both searches start at the green cell and look for the red cell. Blue cells have been expanded. Compare how many cells each algorithm expands before it finds a path of the same length. Run it in the [web version of the lecture](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-02/lecture.html#anim-grid).


![Expanded cells of breadth-first search and A* with the Manhattan heuristic on the same grid. Both return a shortest path, but A* expands far fewer cells [3].](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-02/figures/w02_fig2.png)

*Figure 2.2. Expanded cells of breadth-first search and A* with the Manhattan heuristic on the same grid. Both return a shortest path, but A* expands far fewer cells [3].*




<details>
<summary><b>Check your understanding.</b> Which heuristic is admissible for a robot that moves on a grid with four moves of cost 1?</summary>

A. Twice the Manhattan distance  
B. The Manhattan distance  
C. The number of obstacles on the map  
D. The square of the Euclidean distance

**Answer: B.** The Manhattan distance is the length of a path without obstacles, so it can never exceed the true remaining cost.

</details>

## Path planning for vehicles and machines

Path planning in engineering adds physical constraints to graph search. Occupancy grids and road graphs give the search space, while costs encode travel time, energy, risk or wear [4]. Vehicles cannot turn on the spot, so planners for cars and trucks search over positions and headings. Hybrid A* keeps continuous positions inside discrete cells and has been used to park and manoeuvre autonomous vehicles in unstructured environments such as parking lots [9]. Surveys of self-driving urban vehicles place such graph-search planners next to sampling-based and optimisation-based methods [10].

Terrain adds cost and feasibility. A haul road in a mine, a pipeline or a forest road must respect a maximum grade, and steep segments cost more fuel and time. A digital elevation model turns the terrain into a grid of heights, and the cost of moving between two cells can combine distance with a penalty on slope and a hard limit above which the move is forbidden. The notebook applies this idea to a real elevation grid near Isparta from the Copernicus GLO-90 model, served by the Open-Meteo Elevation API [5, 11].

## Beyond systematic search: Local search

Systematic search remembers the paths it explores. Local search keeps only a current solution and improves it by small changes, which suits problems where the path does not matter and only the final configuration counts, such as a layout or a schedule. Hill climbing accepts only improvements and stops at the first local optimum. Simulated annealing sometimes accepts worse solutions, with a probability that falls as a temperature parameter is lowered, and can therefore escape local optima [12]. Weeks 5 and 6 develop this line into evolutionary and swarm algorithms, which keep a population of candidate solutions instead of one.


<details>
<summary><b>Check your understanding.</b> Why can simulated annealing escape a local optimum where hill climbing stops?</summary>

A. It evaluates every possible solution  
B. It sometimes accepts a worse solution, with a probability that decreases over time  
C. It uses the gradient of the objective  
D. It always restarts from the best known solution

**Answer: B.** Occasional uphill moves in cost let the search leave the basin of a local optimum early in the run.

</details>

# Hands-on lab

The notebook implements breadth-first search, uniform-cost search and A* from scratch on occupancy grids, counts expanded cells for different heuristics, and then plans a route across real terrain near Isparta. The elevation grid comes from the Copernicus GLO-90 digital elevation model through the Open-Meteo Elevation API [5, 11], and the cost of each move combines distance, a slope penalty and a grade limit.

## 0. Setup

In [ ]:
import heapq
import json
import math
import random
import urllib.request
from collections import deque

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

## Python step 2: Lists, tuples and loops

This part follows the Python step at the end of the lecture. Run the cells in order, compare each result with the explanation, and change values to see what happens. The exercises at the end of the step give immediate feedback.

### Lists

A list holds several values in a fixed order and is written in square brackets. Positions are counted from 0, so `heights_m[0]` is the first element, and negative positions count from the end. A slice such as `heights_m[2:5]` returns the elements at positions 2, 3 and 4: The start is included and the stop is excluded. Lists can grow with `append`, and `len`, `min`, `max` and `sum` work on any list of numbers. The list below holds the ground elevation of successive cells along a planned route.

In [ ]:
heights_m = [1012, 1018, 1031, 1047, 1040, 1066, 1080]
print(len(heights_m), heights_m[0], heights_m[-1])
print(heights_m[2:5])
heights_m.append(1093)
print(max(heights_m) - min(heights_m), "m between the lowest and the highest cell")

### Loops over lists

A `for` loop can run over the elements of a list directly or over positions produced by `range`. The loop below adds up all rises between neighbouring cells, which gives the total climb that a vehicle must make. The operator `+=` adds to a variable in place. The function `enumerate` supplies position and value together, which is often clearer than indexing by hand.

In [ ]:
total_climb = 0
for i in range(1, len(heights_m)):
    rise = heights_m[i] - heights_m[i - 1]
    if rise > 0:
        total_climb += rise
print("total climb:", total_climb, "m")
for i, h in enumerate(heights_m[:3]):
    print("cell", i, "is at", h, "m")

### Grids and tuples

A map for path planning can be written as a list of strings, one string per row, with `#` marking obstacles. `grid[r][c]` then reads the character in row r and column c. A cell is naturally described by a tuple `(row, column)`. Tuples are written with parentheses and, unlike lists, cannot be changed after creation, which suits fixed coordinates. Two nested loops visit every cell, and unpacking assigns the parts of a tuple to separate names.

In [ ]:
grid = ["S..#....",
        ".#.#.##.",
        ".#...#..",
        "...#...G"]
rows, cols = len(grid), len(grid[0])
for r in range(rows):
    for c in range(cols):
        if grid[r][c] == "S":
            start = (r, c)
        elif grid[r][c] == "G":
            goal = (r, c)
print("start", start, "goal", goal)
r, c = goal
print("the goal is in row", r, "and column", c)

The neighbours of a cell follow from four moves. A move is valid when it stays inside the map and does not enter an obstacle, and every valid neighbour is appended to a list.

In [ ]:
moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]    # up, down, left, right
cell = (1, 2)
free_neighbours = []
for dr, dc in moves:
    nr, nc = cell[0] + dr, cell[1] + dc
    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != "#":
        free_neighbours.append((nr, nc))
print(free_neighbours)

### While loops, sets and a queue

A `while` loop repeats as long as its condition holds, which suits searches whose length is not known in advance. Breadth-first search keeps a queue of cells to explore: New cells join at the back, and the next cell to expand leaves from the front. `collections.deque` provides such a queue with `append` and `popleft`. A set, written with braces, stores the cells already seen and answers membership questions such as `nxt not in visited` quickly. A dictionary `parent` records from which cell each cell was reached; Week 3 explains dictionaries in detail. The statement `break` leaves a loop early.

In [ ]:
from collections import deque

frontier = deque([start])
visited = {start}
parent = {start: None}
while frontier:                 # an empty queue counts as False
    cell = frontier.popleft()
    if cell == goal:
        break
    for dr, dc in moves:
        nxt = (cell[0] + dr, cell[1] + dc)
        inside = 0 <= nxt[0] < rows and 0 <= nxt[1] < cols
        if inside and grid[nxt[0]][nxt[1]] != "#" and nxt not in visited:
            visited.add(nxt)
            parent[nxt] = cell
            frontier.append(nxt)

path = []
while cell is not None:
    path.append(cell)
    cell = parent[cell]
path.reverse()
print(len(path) - 1, "moves:", path)

The second `while` loop walks back from the goal through the recorded parents until it reaches the start, whose parent is `None`, and `reverse` puts the path in travel order. The notebook of this week extends the same pattern with a priority queue for A*.

**Quick check.** After heights = [5, 7, 9, 11], what does print(heights[1:3], heights[-1]) display?

A. [7, 9] 11  
B. [5, 7, 9] 11  
C. [7, 9, 11] 5  
D. [7, 9] 9

<details><summary>Show the answer</summary>

**A.** The slice includes position 1 and stops before position 3, and position -1 is the last element.

</details>

### Exercises for Python step 2

**Exercise 2.1.** Use two nested loops to count the cells of `grid` that are not obstacles, including the start and the goal, and store the number in `py_free_cells`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py2_free_cells_reference():
    g = ["S..#....", ".#.#.##.", ".#...#..", "...#...G"]
    return sum(1 for row in g for ch in row if ch != "#")

In [ ]:
py_free_cells = None
check("Exercise 2.1", py_free_cells, _py2_free_cells_reference())

**Exercise 2.2.** Count the neighbouring pairs of cells in `heights_m` (with 1093 appended) where the ground rises by more than 12 m, and store the count in `py_steep_rises`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py2_steep_rises_reference():
    h = [1012, 1018, 1031, 1047, 1040, 1066, 1080, 1093]
    return sum(1 for i in range(1, len(h)) if h[i] - h[i - 1] > 12)

In [ ]:
py_steep_rises = None
check("Exercise 2.2", py_steep_rises, _py2_steep_rises_reference())

**Exercise 2.3.** A new obstacle blocks row 3, column 5. Copy the search cell above, run it on `grid2` instead of `grid`, and store the number of moves of the new shortest path in `py_moves`.

In [ ]:
grid2 = ["S..#....",
         ".#.#.##.",
         ".#...#..",
         "...#.#.G"]

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py2_moves_reference():
    g = ["S..#....", ".#.#.##.", ".#...#..", "...#.#.G"]
    from collections import deque
    s, t = (0, 0), (3, 7)
    q, par = deque([s]), {s: None}
    while q:
        a = q.popleft()
        if a == t:
            break
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            b = (a[0] + dr, a[1] + dc)
            if 0 <= b[0] < 4 and 0 <= b[1] < 8 and g[b[0]][b[1]] != "#" and b not in par:
                par[b] = a
                q.append(b)
    n, a = 0, t
    while par[a] is not None:
        a, n = par[a], n + 1
    return n

In [ ]:
py_moves = None
check("Exercise 2.3", py_moves, _py2_moves_reference())

## 1. A factory floor as an occupancy grid

**Python move: lists of strings.** Each string is one row of the map: `.` is free floor, `#` is a machine or wall, `S` the start and `G` the goal. `grid[r][c]` reads the character in row `r` and column `c`.

In [ ]:
FLOOR = [
    "....................",
    "..S.......#.........",
    "..........#.........",
    "...####...#...####..",
    "..........#.........",
    "..........#.....#...",
    "......#.........#...",
    "......#####.....#...",
    "..........#.......G.",
    "....................",
]


def find(grid, ch):
    for r, row in enumerate(grid):
        c = row.find(ch)
        if c >= 0:
            return (r, c)


start, goal = find(FLOOR, "S"), find(FLOOR, "G")
print("start", start, "goal", goal)

## 2. Neighbours and breadth-first search

**Python move: functions, tuples, sets and deque.** `neighbours` returns the free cells around a cell. Breadth-first search keeps a first-in first-out queue, a `parent` dictionary that records how each cell was reached, and counts how many cells it expands.

In [ ]:
MOVES = [(1, 0), (-1, 0), (0, 1), (0, -1)]


def neighbours(cell, grid):
    r, c = cell
    result = []
    for dr, dc in MOVES:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] != "#":
            result.append((nr, nc))
    return result


def bfs(grid, start, goal):
    frontier, parent, expanded = deque([start]), {start: None}, 0
    while frontier:
        cell = frontier.popleft()
        expanded += 1
        if cell == goal:
            break
        for nxt in neighbours(cell, grid):
            if nxt not in parent:
                parent[nxt] = cell
                frontier.append(nxt)
    return parent, expanded


def path_from(parent, goal):
    path, cell = [], goal
    while cell is not None:
        path.append(cell)
        cell = parent[cell]
    return path[::-1]          # reverse: start first


parent, n_bfs = bfs(FLOOR, start, goal)
path_bfs = path_from(parent, goal)
print(f"BFS: {len(path_bfs) - 1} steps, {n_bfs} cells expanded")

**Python move: plotting a grid.** `imshow` shows a table of numbers as an image. The map is converted into zeros and ones with a nested list comprehension, a compact loop inside brackets.

In [ ]:
def show(grid, path=None, title=""):
    img = [[1 if ch == "#" else 0 for ch in row] for row in grid]
    plt.figure(figsize=(6, 3))
    plt.imshow(img, cmap="Greys")
    if path:
        plt.plot([c for r, c in path], [r for r, c in path], lw=3)
    plt.xticks([]); plt.yticks([]); plt.title(title); plt.show()


show(FLOOR, path_bfs, "Breadth-first search")

## 3. A* with a priority queue

**Python move: heapq and default arguments.** `heapq.heappush(heap, (priority, item))` inserts an item, and `heapq.heappop(heap)` removes the one with the smallest priority. The parameter `weight=1.0` has a default value, so `astar(grid, s, g)` and `astar(grid, s, g, weight=2)` both work.

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(grid, start, goal, h=manhattan, weight=1.0, step_cost=lambda cell: 1.0):
    g, parent, closed, expanded = {start: 0.0}, {start: None}, set(), 0
    heap, tie = [(weight * h(start, goal), 0, start)], 0
    while heap:
        f, _, cell = heapq.heappop(heap)
        if cell in closed:
            continue
        closed.add(cell)
        expanded += 1
        if cell == goal:
            break
        for nxt in neighbours(cell, grid):
            new_g = g[cell] + step_cost(nxt)
            if nxt not in g or new_g < g[nxt]:
                g[nxt], parent[nxt] = new_g, cell
                tie += 1
                heapq.heappush(heap, (new_g + weight * h(nxt, goal), tie, nxt))
    return parent, expanded, g.get(goal)


zero = lambda a, b: 0
for name, h, w in [("uniform cost (h = 0)", zero, 1), ("A*, Manhattan", manhattan, 1), ("weighted A*, w = 3", manhattan, 3)]:
    par, n, cost = astar(FLOOR, start, goal, h=h, weight=w)
    print(f"{name:22s} cost {cost:4.0f}   expanded {n:4d}")

### Your turn, exercise 1

The Euclidean distance is also admissible for four moves, but it is smaller than the Manhattan distance. Write `euclid(a, b)`, run A* with it and store the number of expanded cells in `n_euclid`. Is it larger or smaller than with the Manhattan heuristic, and why?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _n_euclid_reference():
    return astar(FLOOR, start, goal, h=lambda a, b: math.hypot(a[0] - b[0], a[1] - b[1]))[1]

In [ ]:
n_euclid = None   # your computation
check("Exercise 1", n_euclid, _n_euclid_reference())

## 4. Real terrain near Isparta

A 20 x 20 grid of points covers about 11 km by 11 km west of the city. The Open-Meteo Elevation API returns heights from the Copernicus GLO-90 model for up to 100 coordinates per request [5, 11]. When the service cannot be reached, a synthetic valley with two ridges is generated instead.

**Python move: NumPy arrays, first contact.** `np.array` turns a list of lists into an array that supports arithmetic on all cells at once. Week 4 introduces NumPy properly.

In [ ]:
N = 20
lats = [37.72 + 0.10 * i / (N - 1) for i in range(N)]
lons = [30.42 + 0.125 * j / (N - 1) for j in range(N)]
points = [(la, lo) for la in lats for lo in lons]
heights = []
try:
    for k in range(0, len(points), 100):
        chunk = points[k:k + 100]
        url = ("https://api.open-meteo.com/v1/elevation?latitude=" + ",".join(f"{p[0]:.5f}" for p in chunk)
               + "&longitude=" + ",".join(f"{p[1]:.5f}" for p in chunk))
        with urllib.request.urlopen(url, timeout=20) as r:
            heights += json.load(r)["elevation"]
    Z = np.array(heights).reshape(N, N)[::-1]          # north at the top
    terrain_source = "Copernicus GLO-90 via Open-Meteo"
except Exception as error:
    print("Elevation download failed, using a synthetic valley:", type(error).__name__)
    y, x = np.mgrid[0:1:N * 1j, 0:1:N * 1j]
    Z = 1050 + 350 * np.exp(-((x - 0.5) ** 2) / 0.03) * (0.6 + 0.4 * np.sin(6 * y)) + 200 * np.exp(-((y - 0.8) ** 2) / 0.02)
    terrain_source = "synthetic valley (fallback)"
CELL = 11000 / (N - 1)        # metres between neighbouring points
print(terrain_source, "| height range", round(float(Z.min())), "to", round(float(Z.max())), "m")
plt.figure(figsize=(4.6, 4)); plt.imshow(Z, cmap="terrain"); plt.colorbar(label="m"); plt.title("Elevation grid"); plt.show()

## 5. Terrain-aware A*

Moves go to the eight neighbouring cells. A move costs its horizontal length multiplied by `1 + penalty * grade`, and moves steeper than `max_grade` are forbidden. The straight-line distance to the goal stays admissible, because no move costs less than its length.

In [ ]:
def terrain_astar(Z, start, goal, penalty=10.0, max_grade=0.25):
    rows, cols = Z.shape
    h = lambda a: CELL * math.hypot(a[0] - goal[0], a[1] - goal[1])
    g, parent, closed, heap, tie, expanded = {start: 0.0}, {start: None}, set(), [(h(start), 0, start)], 0, 0
    while heap:
        _, _, cell = heapq.heappop(heap)
        if cell in closed:
            continue
        closed.add(cell); expanded += 1
        if cell == goal:
            break
        r, c = cell
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                nr, nc = r + dr, c + dc
                if (dr or dc) and 0 <= nr < rows and 0 <= nc < cols:
                    dist = CELL * math.hypot(dr, dc)
                    grade = abs(Z[nr, nc] - Z[r, c]) / dist
                    if grade > max_grade:
                        continue
                    new_g = g[cell] + dist * (1 + penalty * grade)
                    if (nr, nc) not in g or new_g < g[(nr, nc)]:
                        g[(nr, nc)], parent[(nr, nc)] = new_g, cell
                        tie += 1
                        heapq.heappush(heap, (new_g + h((nr, nc)), tie, (nr, nc)))
    if goal not in parent:
        return None, expanded
    return path_from(parent, goal), expanded


def describe(path):
    length = sum(CELL * math.hypot(a[0] - b[0], a[1] - b[1]) for a, b in zip(path, path[1:]))
    climb = sum(max(0.0, Z[b] - Z[a]) for a, b in zip(path, path[1:]))
    return length / 1000, climb


S2, G2 = (N // 2, 0), (N // 2, N - 1)
plt.figure(figsize=(5, 4.3)); plt.imshow(Z, cmap="terrain")
for pen in [0, 5, 30]:
    route, n_exp = terrain_astar(Z, S2, G2, penalty=pen)
    if route:
        L, climb = describe(route)
        plt.plot([c for r, c in route], [r for r, c in route], lw=2.5, label=f"penalty {pen}: {L:.1f} km, climb {climb:.0f} m")
plt.legend(fontsize=7, loc="lower left"); plt.title(terrain_source, fontsize=9); plt.show()

### Your turn, exercise 2

Find the smallest grade limit, as a whole percentage between 1 and 60, for which a route from `S2` to `G2` exists with penalty 10. Store it as a fraction in `min_grade`, for example 0.12 for 12 percent. Use a `for` loop that stops with `break` at the first feasible value.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _min_grade_reference():
    for p in range(1, 61):
        if terrain_astar(Z, S2, G2, penalty=10, max_grade=p / 100)[0] is not None:
            return p / 100

In [ ]:
min_grade = None   # your computation
check("Exercise 2", min_grade, _min_grade_reference())

### Your turn, exercise 3

An engineer claims that multiplying the straight-line heuristic by 3 speeds up the terrain planner without changing the route. Test the claim: Write a weighted version of the heuristic, compare expanded cells and route cost for penalty 10, and set `claim_holds` to `True` or `False`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _claim_reference():
    return False   # weighted A* expands fewer cells but may return a costlier route; the claim does not hold in general

In [ ]:
claim_holds = None   # True or False, after your experiment
check("Exercise 3", claim_holds, _claim_reference())

## Discipline challenges

Each row names a search problem from one department. Write its formulation and propose a heuristic, then decide whether systematic search, weighted search or local search suits its size.

| Department | Challenge |
|---|---|
| Mining Engineering | Haul-road routing between a loading point and the crusher with a maximum grade and a fuel cost per metre of climb. |
| Civil Engineering | Corridor selection for a road or pipeline across a digital elevation model with slope and land-use costs. |
| Industrial Engineering | Order of picking items in a warehouse aisle system to minimise walking distance. |
| Electrical and Electronics Engineering | Routing a cable harness or a printed-circuit trace around keep-out zones. |
| Mechanical Engineering | Collision-free path of a robot arm tool point between two fixtures. |
| Automotive Engineering | Parking manoeuvre planning with position and heading states, as in hybrid A*. |
| Computer Engineering | Shortest route of packets in a network with link delays as costs. |
| Geological Engineering and Earth Sciences Engineering | Field survey route that visits outcrops on steep terrain with limited walking time. |
| Chemistry and Chemical Engineering | Reaction route from a starting compound to a target through known transformations with yields as costs. |
| Mathematics | Proof that consistency implies admissibility, and a counterexample in which an inadmissible heuristic misleads A*. |

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-02/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Name one decision in your field that is a sequence of actions. What would the states, actions and path cost be?
2. When would an engineer accept a slightly longer path in exchange for a much faster planner, as with weighted A*?
3. Which Python data structure from this week (tuple, dictionary, set, deque, heap) do you expect to use again, and for what?

## Weekly task and submission

Formulate one search problem from your department with states, actions, transition model, goal test and path cost, and justify an admissible heuristic for it in about 400 words. Then run the terrain planner of the notebook with two different slope penalties and report how path length, total climb and expanded cells change. Cite at least three works from this week's references [1, 3, 4].

The weekly task is optional and supports self-learning and a personal portfolio. During an active semester, it can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com. Optional work carries no separate weight, but the instructor may take it into account as a discretionary adjustment of the midterm component of the grade.

## Research and report assignment (optional)

**Path planning in practice.** Review how path planning is done in one application area: autonomous haul trucks in mining, pipeline or road route selection on digital elevation models, or motion planning for automated vehicles. Compare graph-search planners with sampling-based or optimisation-based planners and discuss the constraints of the vehicle and the terrain [4, 9, 10].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com. It carries no separate weight, but the instructor may take it into account as a discretionary adjustment of the midterm component of the grade. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and uses the template in `exams/REPORT_TEMPLATE.md`.

## References

[1] Russell, S., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.

[2] Pearl, J. (1984). *Heuristics: Intelligent Search Strategies for Computer Problem Solving*. Addison-Wesley.

[3] Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). A formal basis for the heuristic determination of minimum cost paths. *IEEE Transactions on Systems Science and Cybernetics*, *4*(2), 100-107. <https://doi.org/10.1109/TSSC.1968.300136>

[4] LaValle, S. M. (2006). *Planning Algorithms*. Cambridge University Press. <https://doi.org/10.1017/CBO9780511546877>

[5] European Space Agency (2021). *Copernicus Global Digital Elevation Model (GLO-90)*. <https://doi.org/10.5270/ESA-c5d3d65>

[6] Python Software Foundation (2026). *The Python Tutorial (Python 3 documentation)*. <https://docs.python.org/3/tutorial/>

[7] Korf, R. E. (1985). Depth-first iterative-deepening: An optimal admissible tree search. *Artificial Intelligence*, *27*(1), 97-109. <https://doi.org/10.1016/0004-3702(85)90084-0>

[8] Dijkstra, E. W. (1959). A note on two problems in connexion with graphs. *Numerische Mathematik*, *1*, 269-271. <https://doi.org/10.1007/BF01386390>

[9] Dolgov, D., Thrun, S., Montemerlo, M., & Diebel, J. (2010). Path planning for autonomous vehicles in unknown semi-structured environments. *The International Journal of Robotics Research*, *29*(5), 485-501. <https://doi.org/10.1177/0278364909359210>

[10] Paden, B., Čáp, M., Yong, S. Z., Yershov, D., & Frazzoli, E. (2016). A survey of motion planning and control techniques for self-driving urban vehicles. *IEEE Transactions on Intelligent Vehicles*, *1*(1), 33-55. <https://doi.org/10.1109/TIV.2016.2578706>

[11] Open-Meteo (2026). *Open-Meteo free weather API: Historical Weather API and Elevation API (data licensed under CC BY 4.0)*. <https://open-meteo.com>

[12] Kirkpatrick, S., Gelatt, C. D., & Vecchi, M. P. (1983). Optimization by simulated annealing. *Science*, *220*(4598), 671-680. <https://doi.org/10.1126/science.220.4598.671>